# 파이프라인 파서 워크벤치샘플 파일 하나로 `BasePipelineParser` 파서를 **개발·검증·내보내기**까지 하는 자리.> 🔴 **이 노트북은 운영 코드를 «부른다». 어떤 단계도 다시 구현하지 않는다.**> 여기서 되는 것이 운영에서 안 되면, 도구가 신뢰받는 바로 그 순간에 거짓말을 한 것이다.> 그래서 읽기(`_read_file_to_dataframe`)도, 정리(`clean_for_postgres`)도, 파서 탐색> (`directory_watcher.scan_workspace_pipeline_parsers`)도, 컬럼 검사도 전부 **운영이 쓰는 그> 함수**다. `pd.read_csv`를 직접 부르는 셀은 이 노트북에 하나도 없다.**파서 작성자가 쓰는 것은 둘뿐이다** — `match()`와 `process_dataframe()`. 나머지는 베이스의 것:    파일 → _read_file_to_dataframe → process_dataframe → clean_for_postgres → list[dict] → DB**여는 법.** 이 박스에는 `jupyter notebook`/`jupyterlab`이 **설치돼 있지 않다**(`ipykernel`·`jupyter_client`는 있다). VS Code/Cursor의 노트북 편집기에서 열고 커널을 **conda `assy_manager`**로 고르면 그대로 돈다. 서버형(`jupyter lab`)으로 열려면 패키지를 새로 설치해야 하고, 그건소유자 판단이라 여기서 하지 않았다.

## 0. 부트스트랩 — 운영과 «같은» import 환경플러그인이 `import pipeline_base`(최상위 이름)로 베이스를 찾을 수 있어야 한다. 그 `sys.path`준비를 여기서 흉내 내지 않고 **운영이 부르는 함수를 그대로 부른다** —`directory_watcher.prepare_plugin_imports()`. `scan_workspace_pipeline_parsers()`가 파서를로드하기 직전에 부르는 바로 그 함수이고(구 이름 `_register_legacy_import_shim`도 같은 객체다),경로 추가와 구식 `server.parsers.*` import 별칭을 **한 번에** 건다. 스스로 멱등이라고 선언돼 있다.🔴 **왜 흉내 내면 안 되나:** 베이스 클래스가 두 모듈 정체성(`pipeline_base` vs`server.parsers.pipeline_base`)으로 로드되면 `issubclass`가 깨져 **로더가 파서를 못 알아본다.**노트북에서만 「파서가 없다」가 되거나, 더 나쁘게는 노트북에서만 «있다»가 된다.

In [ ]:
import os, sys, jsonfrom pathlib import Path# 이 노트북의 위치에서 저장소 뿌리를 찾는다 (이 박스의 절대경로를 박아 넣지 않는다)NB_DIR = Path.cwd()REPO = next(p for p in [NB_DIR, *NB_DIR.parents] if (p / "server" / "parsers").is_dir())SERVER = REPO / "server"if str(SERVER) not in sys.path:    sys.path.insert(0, str(SERVER))       # parsers 패키지에 닿기 위한 최소한의 한 줄from parsers import directory_watcherfrom parsers.directory_watcher import scan_workspace_pipeline_parsers, SCAN_UNAVAILABLE# 운영의 부트스트랩. import만으로는 «안 걸린다» - 이 준비는 호출 시점에 일어난다.# 베이스를 두 이름으로 로드하면 클래스 객체가 둘이 되고 issubclass가 False가 되어,# 로더가 예외 없이 «파서가 없다»고 보고한다. 그래서 이 한 줄은 편의가 아니라 필수다.directory_watcher.prepare_plugin_imports()from pipeline_base import BasePipelineParser   # 플러그인 스크립트와 «같은» 최상위 모듈명import pandas as pdprint("repo   :", REPO)print("base   :", BasePipelineParser.__module__, "->", sys.modules["pipeline_base"].__file__)print("동일 객체:", sys.modules["pipeline_base"].BasePipelineParser is BasePipelineParser)print("pandas :", pd.__version__)

## 1. 내가 고를 것 — **이 셀만 고친다**샘플 데이터는 gitignore 대상(`**/raws/*`, `**/*.csv`)이라 저장소에 없다. 경로는 **당신이** 준다.

In [ ]:
# 대상 테이블(= ingestion_workspace 폴더 이름)TABLE = "inventory_master"# 개발용 샘플 파일 하나. 비워 두면 그 워크스페이스의 raws/에서 가장 최근 파일을 집는다.SAMPLE_FILE = ""# match() 일괄 점검에 쓸 폴더. 비우면 raws/ 전체.SWEEP_DIR = ""WORKSPACE = SERVER / "ingestion_workspace" / TABLESCRIPTS = WORKSPACE / "scripts"RAWS = WORKSPACE / "raws"def _newest(folder):    files = [p for p in Path(folder).rglob("*") if p.is_file()]    return max(files, key=lambda p: p.stat().st_mtime) if files else NoneSAMPLE = Path(SAMPLE_FILE) if SAMPLE_FILE else (_newest(RAWS) if RAWS.is_dir() else None)SWEEP = Path(SWEEP_DIR) if SWEEP_DIR else RAWSassert SAMPLE and SAMPLE.is_file(), f"샘플 파일을 찾지 못했다. SAMPLE_FILE에 경로를 직접 적을 것 (raws={RAWS})"print("workspace:", WORKSPACE if WORKSPACE.is_dir() else f"{WORKSPACE}  [없음]")print("scripts  :", SCRIPTS if SCRIPTS.is_dir() else f"{SCRIPTS}  [없음]")print("sample   :", SAMPLE, f"({SAMPLE.stat().st_size:,} bytes)")print("basename :", BasePipelineParser.get_basename(str(SAMPLE)), "  <- 운영이 보는 «원래» 파일명")

## 2. 이 파일을 «누가» 집는가운영은 `scripts/`의 모든 스크립트를 훑어 **`match()`가 True인 첫 번째** 파서를 쓴다(`_discover_and_execute_pipeline`). 아래는 같은 로더로 훑되 **claim을 하지 않아** 전부를 본다 —「첫 번째」만 보면 **두 파서가 같은 파일을 집는 상태**가 안 보이기 때문이다.

In [ ]:
def claimers(file_path, scripts_dir=None):    """운영 로더로 훑어 (스크립트, 클래스, match 결과, 에러)를 «전부» 모은다. 아무것도 claim하지 않는다."""    rows, load_errors = [], {}    def visit(filename, cls):        try:            hit, err = bool(cls.match(str(file_path))), None        except Exception as e:            hit, err = None, f"{type(e).__name__}: {e}"        rows.append({"script": filename, "class": cls.__name__, "match": hit,                     "error": err, "cls": cls})        return None                      # 절대 claim하지 않는다 = 전수 조사    out = scan_workspace_pipeline_parsers(str(scripts_dir or SCRIPTS), visit, load_errors)    if out is SCAN_UNAVAILABLE:        raise FileNotFoundError(f"scripts 폴더가 없다: {scripts_dir or SCRIPTS}")    return rows, load_errorsrows, load_errors = claimers(SAMPLE)for e_file, tb in load_errors.items():    print(f"[로드 실패] {e_file}: {tb.strip().splitlines()[-1]}")if not rows:    print("scripts/에 BasePipelineParser 서브클래스가 없다.")for r in rows:    mark = {True: "[집는다]", False: "[  -  ]", None: "[예외!]"}[r["match"]]    print(f"  {mark} {r['script']}::{r['class']}" + (f"  {r['error']}" if r["error"] else ""))winners = [r for r in rows if r["match"]]PARSER_CLS = winners[0]["cls"] if winners else Noneif len(winners) > 1:    print(f"\n[경고] {len(winners)}개가 같은 파일을 집는다 - 운영은 «첫 번째»"          f"({winners[0]['class']})를 쓴다. 나머지는 이 파일에 대해 영원히 안 돈다.")print("\n선택된 파서:", PARSER_CLS.__name__ if PARSER_CLS else "없음 (std parser 폴백 대상)")

## 3. 세 단계를 «따로» 본다무엇이 내 가정을 깼는지 보려면 단계가 갈려 있어야 한다. `run_stages()`는 운영이 부르는 세메서드를 그 순서로 부르고, 운영이 `parse()` 직전에 붙이는 속성(`rel_path`·`source_root`)까지**같이 붙인다** — 그걸 읽는 파서가 여기서만 다르게 동작하지 않도록.

In [ ]:
from types import SimpleNamespacedef run_stages(parser_cls, file_path, rel_path=None, source_root=None):    """운영과 «같은» 순서로 같은 메서드를 부른다: read -> process -> clean."""    p = parser_cls()    p.rel_path = rel_path                    # 운영이 parse() 전에 붙이는 속성(_claim_first_match)    p.source_root = source_root    raw = p._read_file_to_dataframe(str(file_path))       # 베이스의 읽기(오버라이드했다면 그것)    processed = p.process_dataframe(raw.copy(deep=True))  # 작성자가 쓰는 부분    records = p.clean_for_postgres(processed)             # DB로 갈 list[dict]    return SimpleNamespace(parser=p, raw=raw, processed=processed, records=records)S = run_stages(PARSER_CLS, SAMPLE)print(f"1 raw       {S.raw.shape[0]:,} rows x {S.raw.shape[1]} cols")print(f"2 processed {S.processed.shape[0]:,} rows x {S.processed.shape[1]} cols")print(f"3 records   {len(S.records):,} dicts")S.raw.head()

In [ ]:
# 2단계가 «무엇을 바꿨나» - 컬럼과 dtype의 차이만 본다before, after = S.raw, S.processedadded = [c for c in after.columns if c not in before.columns]removed = [c for c in before.columns if c not in after.columns]retyped = [(c, str(before[c].dtype), str(after[c].dtype)) for c in before.columns           if c in after.columns and before[c].dtype != after[c].dtype]print("추가된 컬럼:", added or "없음")print("사라진 컬럼:", removed or "없음")print("타입 바뀜  :", retyped or "없음")print("행 수 변화 :", f"{len(before):,} -> {len(after):,}"      + ("   <- 행이 바뀐다면 의도한 것인지 확인" if len(before) != len(after) else ""))after.head()

In [ ]:
# 3단계: 인서터가 실제로 받는 것. 여기 보이는 «값의 타입»이 그대로 DB로 간다.import collectionsif S.records:    print("첫 레코드:")    for k, v in S.records[0].items():        print(f"   {k:24s} {type(v).__name__:10s} {v!r}"[:120])    types = collections.Counter(type(v).__name__ for rec in S.records[:2000] for v in rec.values())    print("\n값 타입 분포(앞 2000행):", dict(types))    odd = set(types) - {"str", "int", "float", "bool", "NoneType", "Timestamp", "datetime", "date"}    if odd:        print(f"[경고] JSON이 모르는 타입이 섞여 있다: {odd} - process_dataframe에서 캐스팅할 것")else:    print("레코드가 0건이다.")

## 4. 여기서 고치고, 여기서 다시 돌린다아래 `PARSER_SOURCE`를 고쳐 실행하면 **이후 셀 전부**가 새 파서로 채점된다. 출발 모양은`server/parsers/custom_parser.py.sample`과 같다 — 예시가 둘로 갈라지지 않도록.🔴 **파서를 «문자열»로 쓰는 이유는 하나다: 이 글자들이 그대로 배포되기 때문이다.** 셀에 보통클래스로 쓰고 나중에 `inspect.getsource`로 긁어 내보내는 방법도 있는데, 그건 **마지막으로실행된** 것을 긁어 오므로 화면과 다른 것이 나갈 수 있다. 여기서는 **7번 셀이 이 문자열을 그대로파일에 쓴다** — 「노트북에서 된 것」과 「배포된 것」이 정의상 같은 글자다. 대가는 편집기의 문법강조를 잃는 것이고, 그 대신 옮겨 적기가 사라진다.

In [ ]:
PARSER_SOURCE = r"""# 작업 중인 파서. 고칠 것은 match()와 process_dataframe() 둘뿐이다.class DraftParser(BasePipelineParser):    @classmethod    def match(cls, file_path: str) -> bool:        # get_basename: 업로드 접두(user(...)_)와 8자리 hex 접미가 제거된 «원래» 이름        name = BasePipelineParser.get_basename(file_path).lower()        return name.endswith(".csv")    def process_dataframe(self, df: pd.DataFrame) -> pd.DataFrame:        # 예: df["PROD_LINE"] = 1        return df    # 읽기 규칙이 다르면(구분자·인코딩·헤더 위치) 이것만 오버라이드한다:    # def _read_file_to_dataframe(self, file_path: str) -> pd.DataFrame:    #     return pd.read_table(file_path, sep="\t", encoding="cp949")"""_ns = {"BasePipelineParser": BasePipelineParser, "pd": pd}exec(compile(PARSER_SOURCE, "<PARSER_SOURCE>", "exec"), _ns)DRAFT = [v for v in _ns.values()         if isinstance(v, type) and issubclass(v, BasePipelineParser)         and v is not BasePipelineParser]assert len(DRAFT) == 1, f"PARSER_SOURCE에 파서 클래스가 {len(DRAFT)}개다 - 하나만 둘 것"DraftParser = DRAFT[0]D = run_stages(DraftParser, SAMPLE)print(f"{DraftParser.__name__}: 1 {D.raw.shape}   2 {D.processed.shape}   3 {len(D.records):,} dicts")D.processed.head()

## 5. `match()`를 «여러 파일에» 태운다파서가 조용히 실패하는 자리가 여기다 — 너무 많이 집거나(남의 파일을 가로챈다), 너무 적게집거나(내 파일이 std 파서로 새어 나간다). 둘 다 파일이 엉뚱한 테이블에 앉기 전까지 안 보인다.

In [ ]:
CANDIDATES = [DraftParser] + [r["cls"] for r in rows]    # 초안 + scripts/에 이미 있는 것 전부SHOW = 40                                                # 한 줄씩 볼 파일 수 상한files = sorted(p for p in Path(SWEEP).rglob("*") if p.is_file())print(f"{SWEEP} - 파일 {len(files)}개\n")none_claimed, multi_claimed = [], []for n, f in enumerate(files):    hits = []    for cls in CANDIDATES:        try:            if cls.match(str(f)):                hits.append(cls.__name__)        except Exception as e:            hits.append(f"{cls.__name__}!{type(e).__name__}")    if not hits:        none_claimed.append(f)    elif len(hits) > 1:        multi_claimed.append((f, hits))    if n < SHOW:        print(f"  {'O' if hits else '.'} {str(f.relative_to(SWEEP)):58s} {', '.join(hits) or '-'}")if len(files) > SHOW:    print(f"  ... 그리고 {len(files) - SHOW}개 더 (SHOW를 올리면 다 보인다)")print(f"\n아무도 안 집는 파일 {len(none_claimed)}개 / 둘 이상이 집는 파일 {len(multi_claimed)}개")for f, hits in multi_claimed[:20]:    print(f"  [경고] {f.name}: {hits}  -> 운영은 로드 순서상 «첫 번째»만 쓴다")if len(multi_claimed) > 20:    print(f"  ... 그리고 {len(multi_claimed) - 20}개 더")

## 6. 출력 컬럼을 «진짜» 대상 테이블에 대 본다운영의 판정 규칙 그대로다 — 업서트 fast path는 `table.c.get(key)`가 `None`이면`unknown_column`으로 그 경로를 거절한다(`database/crud.py`). 여기서 찾으면 1분, 운영에서 찾으면왕복이다. DB 접속은 필요 없다.

In [ ]:
import pathsfrom database.models import init_dynamic_models, DYNAMIC_TABLEStable_config = json.loads(Path(paths.config_path("table_config.json")).read_text(encoding="utf-8"))init_dynamic_models(table_config)                  # DB 접속 없이 «진짜» Table 객체를 만든다model = DYNAMIC_TABLES.get(TABLE)assert model is not None, f"table_config.json에 '{TABLE}'가 없다"table = model.__table__declared = {c.name for c in table.c}produced = sorted({k for rec in D.records for k in rec})unknown = [k for k in produced if table.c.get(k) is None]      # 운영과 «같은» 술어print(f"대상 테이블 {TABLE} - 컬럼 {len(declared)}개")print("파서가 내는 키:", produced)if unknown:    print(f"\n[경고] 이 테이블에 «없는» 키 {len(unknown)}개 -> 업서트가 fast path를 거절한다: {unknown}")else:    print("\n[OK] 모든 키가 대상 테이블의 컬럼이다")system_cols = {"row_id", "created_at", "updated_at", "is_graph_synced",               "needs_graph_rollback", "graph_synced_at", "business_key_val"}never_filled = sorted(declared - set(produced) - system_cols)print("\n파서가 «안 채우는» 업무 컬럼:", never_filled or "없음", "  (의도한 것인지만 확인)")

## 7. 내보내기 — 그리고 **내보낸 파일로 다시 채점한다**손으로 옮겨 적는 순간 드리프트가 들어온다. 그래서 내보낸 뒤 **운영 로더로 다시 읽어** 같은 행이나오는지 확인한다. 이 확인이 초록이어야 「노트북에서 된 것」이 곧 「배포되는 것」이다.⚠️ `inspect.getsource`는 **마지막으로 «실행된»** 셀의 소스를 준다. 셀을 고치고 실행하지 않은 채내보내면 화면과 다른 것이 나가고, 그 경우 아래 확인 셀이 빨개진다.

In [ ]:
OUT_NAME = "draft_parser.py"        # scripts/에 만들 파일 이름OVERWRITE = False                   # 기존 파일을 덮어쓸 것인가header = ("import pandas as pd\n"          "from pipeline_base import BasePipelineParser\n"          "# server/parsers/notebooks/parser_workbench.ipynb 에서 생성됨\n")body = header + PARSER_SOURCE       # 4번 셀이 «실행한» 바로 그 글자target = SCRIPTS / OUT_NAMESCRIPTS.mkdir(parents=True, exist_ok=True)if target.exists() and not OVERWRITE:    raise FileExistsError(f"{target} 가 이미 있다. OVERWRITE=True로 두거나 OUT_NAME을 바꿀 것")target.write_text(body, encoding="utf-8")print("작성:", target, f"({len(body)} bytes)")print("\n[주의] 이 폴더는 운영이 읽는 자리다. 워처가 도는 중이면 다음 스캔부터 이 파서가 "      "«실제로» 파일을 집는다 - match()를 좁혀 두지 않았다면 남의 파일까지 가져간다(5번 셀).\n")print(body)

In [ ]:
# 내보낸 파일을 «운영 로더»로 다시 읽어 같은 결과인지 확인한다reloaded, errs = claimers(SAMPLE)mine = [r for r in reloaded if r["script"] == OUT_NAME]assert mine, f"{OUT_NAME}가 로더에 안 잡힌다: {errs.get(OUT_NAME, '')}"cls = mine[0]["cls"]R = run_stages(cls, SAMPLE)same_cols = list(R.processed.columns) == list(D.processed.columns)same_rows = R.records == D.recordsprint(f"로드됨      : {OUT_NAME}::{cls.__name__}")print(f"match()     : {mine[0]['match']}")print(f"컬럼 동일   : {same_cols}")print(f"레코드 동일 : {same_rows}  ({len(R.records):,} vs {len(D.records):,})")if same_cols and same_rows:    print("\n[OK] 내보낸 파일이 노트북의 초안과 «같은 행»을 낸다 - 그대로 배포 가능")else:    print("\n[경고] 다르다. 4번 셀을 고치고 «실행하지 않은» 채로 내보냈을 가능성이 가장 크다 - "          "4번 셀을 다시 실행하고 다시 내보낼 것")